# Hierarchical Model for Energy Complex: Crude, Gasoline, Heating Oil

## Learning Objectives

By completing this notebook, you will:
1. Model multiple related commodities with hierarchical structures
2. Implement partial pooling for shared parameters across energy products
3. Capture crack spreads and refining margins in Bayesian framework
4. Forecast energy complex jointly vs independently
5. Quantify information sharing benefits across related commodities

## Prerequisites
- Hierarchical modeling fundamentals (Module 4.1)
- PyMC basics (Module 1)
- Commodity relationships understanding

## Estimated Time: 85 minutes

---

In [ ]:
learning_objectives(["Model multiple related commodities with hierarchical structures", "Implement partial pooling for shared parameters across energy products", "Capture crack spreads and refining margins in Bayesian framework", "Forecast energy complex jointly vs independently", "Quantify information sharing benefits across related commodities", "Hierarchical modeling fundamentals (Module 4.1)"])

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

apply_course_theme()
apply_plot_theme()

In [ ]:
# Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pymc as pm
import arviz as az
import yfinance as yf
from scipy import stats

np.random.seed(42)
az.style.use('arviz-darkgrid')
plt.rcParams['figure.figsize'] = (14, 8)

print("Environment ready!")

## 1. Energy Complex Overview

**Key relationships:**
- **Crude oil** (CL): Input feedstock
- **Gasoline** (RB): Refined product from crude
- **Heating oil** (HO): Refined product from crude

**Crack spreads:**
$$\text{Crack Spread} = P_{\text{product}} - k \cdot P_{\text{crude}}$$

where $k$ is refining ratio (typically 1 barrel crude → multiple products)

**Why hierarchical modeling?**
1. Products share common driver (crude prices)
2. Refining margins have common economic factors
3. Partial pooling improves estimates for less-liquid products
4. Joint forecasting respects economic constraints

In [ ]:
# Fetch energy complex data
tickers = {
    'CL=F': 'Crude',
    'RB=F': 'Gasoline', 
    'HO=F': 'Heating Oil'
}

start = '2020-01-01'
end = '2024-01-01'

prices_dict = {}
for ticker, name in tickers.items():
    data = yf.download(ticker, start=start, end=end, progress=False)
    prices_dict[name] = data['Adj Close']

prices_df = pd.DataFrame(prices_dict).dropna()

print(f"Loaded {len(prices_df)} observations")
print(prices_df.describe())

In [ ]:
# Visualize price series
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

for ax, col in zip(axes, prices_df.columns):
    ax.plot(prices_df.index, prices_df[col], linewidth=1.5)
    ax.set_ylabel(f'{col} Price', fontsize=11)
    ax.set_title(col, fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Date', fontsize=12)
plt.suptitle('Energy Complex: Price Series', fontsize=14, y=0.995)
plt.tight_layout()
plt.show()

In [ ]:
# Compute returns
returns_df = np.log(prices_df / prices_df.shift(1)).dropna() * 100

# Correlation matrix
corr_matrix = returns_df.corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm', 
           center=0, vmin=-1, vmax=1, square=True, ax=ax)
ax.set_title('Energy Complex: Return Correlations', fontsize=14)
plt.tight_layout()
plt.show()

print("\nReturn correlations:")
print(corr_matrix)

**Key insight:** High correlations suggest shared factors

## 2. Crack Spreads Analysis

In [ ]:
# Compute crack spreads (product minus crude)
spreads_df = pd.DataFrame({
    'Gas_Crack': prices_df['Gasoline'] - prices_df['Crude'],
    'HO_Crack': prices_df['Heating Oil'] - prices_df['Crude']
})

# Plot spreads
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

axes[0].plot(spreads_df.index, spreads_df['Gas_Crack'], linewidth=1.5, color='green')
axes[0].axhline(spreads_df['Gas_Crack'].mean(), color='red', linestyle='--', 
               label=f'Mean: ${spreads_df["Gas_Crack"].mean():.2f}')
axes[0].set_ylabel('Spread ($)', fontsize=11)
axes[0].set_title('Gasoline Crack Spread', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(spreads_df.index, spreads_df['HO_Crack'], linewidth=1.5, color='orange')
axes[1].axhline(spreads_df['HO_Crack'].mean(), color='red', linestyle='--',
               label=f'Mean: ${spreads_df["HO_Crack"].mean():.2f}')
axes[1].set_ylabel('Spread ($)', fontsize=11)
axes[1].set_xlabel('Date', fontsize=12)
axes[1].set_title('Heating Oil Crack Spread', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Spread statistics:")
print(spreads_df.describe())

## 3. Hierarchical Return Model

**Model structure:**

$$\begin{align}
r_{i,t} &\sim \mathcal{N}(\mu_i, \sigma_i) && \text{(Individual returns)} \\
\mu_i &\sim \mathcal{N}(\mu_{\text{global}}, \tau) && \text{(Hierarchical mean)} \\
\sigma_i &\sim \text{HalfNormal}(\sigma_{\text{global}}) && \text{(Hierarchical std)}
\end{align}$$

where $i \in \{\text{Crude, Gasoline, Heating Oil}\}$

In [ ]:
# Prepare data for PyMC
n_products = len(returns_df.columns)
product_names = returns_df.columns.tolist()
returns_array = returns_df.values  # Shape: (T, n_products)

with pm.Model() as hierarchical_model:
    
    # Hyperpriors (global parameters)
    mu_global = pm.Normal('mu_global', mu=0, sigma=2)
    tau = pm.HalfNormal('tau', sigma=1)  # Between-product std
    sigma_global = pm.HalfNormal('sigma_global', sigma=3)
    
    # Product-specific parameters (partially pooled)
    mu = pm.Normal('mu', mu=mu_global, sigma=tau, shape=n_products)
    sigma = pm.HalfNormal('sigma', sigma=sigma_global, shape=n_products)
    
    # Likelihood
    for i in range(n_products):
        pm.Normal(f'returns_{product_names[i]}', 
                 mu=mu[i], sigma=sigma[i], 
                 observed=returns_array[:, i])

pm.model_to_graphviz(hierarchical_model)

In [ ]:
# Sample
with hierarchical_model:
    trace_hier = pm.sample(
        draws=1000,
        tune=1000,
        chains=2,
        cores=1,
        random_seed=42,
        return_inferencedata=True
    )

print("\n✅ Sampling complete!")

In [ ]:
# Analyze results
summary = az.summary(trace_hier, var_names=['mu_global', 'tau', 'mu', 'sigma'])
print("Posterior Summary:")
print("="*80)
print(summary)
print("="*80)

In [ ]:
# Compare with empirical estimates
empirical_means = returns_df.mean()
empirical_stds = returns_df.std()

posterior_means = trace_hier.posterior['mu'].mean(dim=('chain', 'draw')).values
posterior_stds = trace_hier.posterior['sigma'].mean(dim=('chain', 'draw')).values

comparison_df = pd.DataFrame({
    'Product': product_names,
    'Empirical Mean': empirical_means.values,
    'Hierarchical Mean': posterior_means,
    'Empirical Std': empirical_stds.values,
    'Hierarchical Std': posterior_stds
})

print("\nEmpirical vs Hierarchical Estimates:")
print(comparison_df)

## 4. Multivariate Model with Correlation Structure

In [ ]:
# Multivariate normal model capturing correlations
with pm.Model() as multivariate_model:
    
    # Mean vector
    mu_vec = pm.Normal('mu', mu=0, sigma=2, shape=n_products)
    
    # Covariance matrix (LKJ prior for correlation)
    sd_dist = pm.HalfNormal.dist(sigma=3, shape=n_products)
    chol, corr, stds = pm.LKJCholeskyCov('chol', n=n_products, eta=2, 
                                         sd_dist=sd_dist, compute_corr=True)
    
    # Likelihood
    pm.MvNormal('returns', mu=mu_vec, chol=chol, observed=returns_array)

pm.model_to_graphviz(multivariate_model)

In [ ]:
# Sample
with multivariate_model:
    trace_mv = pm.sample(
        draws=1000,
        tune=1000,
        chains=2,
        cores=1,
        random_seed=42,
        return_inferencedata=True
    )

print("\n✅ Multivariate sampling complete!")

In [ ]:
# Extract correlation estimates
corr_posterior = trace_mv.posterior['corr'].mean(dim=('chain', 'draw')).values

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Empirical correlation
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm',
           center=0, vmin=-1, vmax=1, square=True, ax=axes[0],
           xticklabels=product_names, yticklabels=product_names)
axes[0].set_title('Empirical Correlation', fontsize=13, fontweight='bold')

# Posterior correlation
sns.heatmap(corr_posterior, annot=True, fmt='.3f', cmap='coolwarm',
           center=0, vmin=-1, vmax=1, square=True, ax=axes[1],
           xticklabels=product_names, yticklabels=product_names)
axes[1].set_title('Posterior Mean Correlation', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Joint Forecasting

Generate correlated forecasts for all three products.

In [ ]:
# Extract posterior samples
mu_samples = trace_mv.posterior['mu'].values.reshape(-1, n_products)
corr_samples = trace_mv.posterior['corr'].values.reshape(-1, n_products, n_products)
std_samples = trace_mv.posterior['stds'].values.reshape(-1, n_products)

# Generate forecasts
n_forecast_samples = 500
forecast_horizon = 20
forecasts = np.zeros((n_forecast_samples, forecast_horizon, n_products))

for i in range(n_forecast_samples):
    # Sample parameters
    idx = np.random.randint(len(mu_samples))
    mu = mu_samples[idx]
    corr = corr_samples[idx]
    stds = std_samples[idx]
    
    # Build covariance matrix
    cov = np.diag(stds) @ corr @ np.diag(stds)
    
    # Generate correlated forecasts
    forecasts[i] = np.random.multivariate_normal(mu, cov, size=forecast_horizon)

print(f"Generated {n_forecast_samples} forecast paths")

In [ ]:
# Plot forecast distributions
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

for i, (ax, product) in enumerate(zip(axes, product_names)):
    # Historical returns (last 50 days)
    ax.plot(range(-50, 0), returns_df[product].values[-50:], 
           'o-', linewidth=1, markersize=3, label='Historical', alpha=0.7)
    
    # Forecast distribution
    forecast_mean = forecasts[:, :, i].mean(axis=0)
    forecast_lower = np.percentile(forecasts[:, :, i], 2.5, axis=0)
    forecast_upper = np.percentile(forecasts[:, :, i], 97.5, axis=0)
    
    forecast_time = range(0, forecast_horizon)
    ax.plot(forecast_time, forecast_mean, 'r-', linewidth=2, label='Forecast Mean')
    ax.fill_between(forecast_time, forecast_lower, forecast_upper,
                    alpha=0.3, color='red', label='95% Interval')
    
    ax.axvline(0, color='black', linestyle='--', linewidth=1, alpha=0.5)
    ax.set_ylabel(f'{product}\nReturn (%)', fontsize=11)
    ax.legend(fontsize=9, loc='upper left')
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Days', fontsize=12)
plt.suptitle('Energy Complex: Joint Return Forecasts', fontsize=14, y=0.995)
plt.tight_layout()
plt.show()

---

## Exercise 1: Compare Pooling Strategies

**Task:** Build three models and compare:
1. **No pooling**: Independent models for each product
2. **Complete pooling**: Single model for all products
3. **Partial pooling**: Hierarchical (already done)

Compare out-of-sample forecast accuracy.

In [ ]:
# YOUR CODE HERE

---

## Exercise 2: Crack Spread Forecasting

**Task:** Model crack spreads directly instead of products.

1. Build hierarchical model for spreads (not returns)
2. Generate spread forecasts
3. Reconstruct product forecasts from crude + spread
4. Compare with direct product forecasting

In [ ]:
# YOUR CODE HERE

---

## Exercise 3: Dynamic Correlation

**Task:** Allow correlation to change over time (DCC-like).

1. Split data into rolling windows
2. Estimate correlation for each window
3. Plot time-varying correlation
4. Model correlation as function of market conditions

In [ ]:
# YOUR CODE HERE

---

## Summary

### Key Takeaways

1. **Energy complex** has strong fundamental relationships
2. **Hierarchical models** capture shared structure across products
3. **Partial pooling** improves estimates via information sharing
4. **Joint forecasting** respects correlations
5. **Crack spreads** reveal refining margins and supply/demand

### Advantages of Hierarchical Approach

- More stable estimates (regularization)
- Better for small-sample products
- Quantifies between vs within variation
- Natural framework for groups

### What's Next

Next notebook: **Grain Complex** (corn, wheat, soybeans) with seasonality

### Additional Resources

- Gelman & Hill (2006): *Data Analysis Using Regression*
- McElreath (2020): *Statistical Rethinking*
- Energy Information Administration (EIA): Crack spread data

---